In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import cv2
import numpy as np
import torch
import gc
import time
import torch
from realesrgan.archs.rrdb_unet_arch import RRDB_UNet
from realesrgan.archs.rrdb_unet_res_arch import RRDB_UNet_res
from realesrgan.archs.rrdb_unet_res2_arch import RRDB_UNet_res2
from realesrgan.archs.rrdb_unet_res3_arch import RRDB_UNet_res3

In [2]:
pad = 8
device = "cuda"
#device = "cpu"

model = RRDB_UNet_res3(
    num_in_ch=3,
    num_out_ch=3,
    highway_channels_base=24,
    processing_channels_base=12,
    num_grow_ch_base = 6,
    ae_rrdb_blocks=3,
    ae_channel_multipliers = [1,2,4,12,36],
    use_attention=True,
    body_rrdb_blocks = 16,
    inference=True
)

In [3]:
model = RRDB_UNet_res3(
    num_in_ch=3,
    num_out_ch=3,
    highway_channels_base=32,
    processing_channels_base=16,
    num_grow_ch_base = 8,
    ae_rrdb_blocks=3,
    ae_channel_multipliers = [1,3,9,24,48],
    use_attention=True,
    body_rrdb_blocks = 24,
    inference=True
)

highway_channels_base 32
processing_channels_base 16
num_grow_ch_base 8
ae_rrdb_blocks 3
ae_channel_multipliers [1, 3, 9, 24, 48]
use_attention True
body_rrdb_blocks 24


model = RRDB_UNet_res3(
    num_in_ch=3,
    num_out_ch=3,
    highway_channels_base=24,
    processing_channels_base=12,
    num_grow_ch_base = 6,
    ae_rrdb_blocks=1,
    ae_channel_multipliers = [1,2,4,8,8],
    use_attention=True,
    body_rrdb_blocks = 4,
    inference=True
)

In [4]:
print(model)

RRDB_UNet_res3(
  (prep): ModuleList(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.01, inplace=True)
  )
  (encoder): ModuleList(
    (0): ModuleList(
      (0-2): 3 x HighwayRRDB(
        (compress): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1))
        (rdb1): ResidualDenseBlock(
          (conv1): Conv2d(16, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv2): Conv2d(24, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv3): Conv2d(32, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv4): Conv2d(40, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv5): Conv2d(48, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (lrelu): LeakyReLU(negative_slope=0.2, inplace=True)
        )
        (rdb2): ResidualDenseBlock(
          (conv1): Conv2d(16, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv2): Conv2d(24, 8, k

In [ ]:
img = cv2.imread("tests/data/lq_4/comic.png")
#img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#img = cv2.resize(img, (4020*1, 4020*4))
img = cv2.resize(img, (4020*1, 4020*1))
#scale = 4
#img = cv2.resize(img, (img.shape[1]*scale,img.shape[0]*scale))
print(img.shape)
import matplotlib.pyplot as plt
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

In [ ]:
# Note: pytorch appears to use different gpu code if you exceed some resolution causing it to be very slow.
# compiling with fixed resolution makes it fast again, but it requires bucketing.
# see esrgan_arch_test2

In [6]:
from realesrgan.real_esrganer_1x import RealESRGANer1x

esrganer = RealESRGANer1x(None, model, pad, device)
#esrganer = RealESRGANer1x("experiments/train_unet_dynamic_iterative_res3_xs/models/net_g_latest.pth", model, pad, device)

test mode - no model weights loaded!


In [12]:
out = esrganer.enhance(img)[0]
out.shape

100%|██████████| 33/33 [00:14<00:00,  2.20it/s]


(4020, 4020, 3)

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))

In [9]:
#exit()

In [10]:
import torch
torch.cuda.empty_cache()